# TradeSense · Component 03 · notebook 38: scorecard of the three bar-level models and the fusion (CPU, minutes)

One table in one format for every model, so a reader can judge each model on **its own task**:
| model | task | metrics that fit the task |
|---|---|---|
| AE-ctx | label-free novelty detection (no pattern labels used) | ROC-AUC and recall on ALL eight families (every family is "unseen" for it), genuine alarm rate |
| XGBoost v2 | classify known manipulation patterns from engineered features | recall and PR-AUC on the six trained families; recall on the two unseen families |
| Transformer v2 | recognise multi-bar sequence patterns | recall and PR-AUC on the six trained families; detection delay |
| Fusion v0 (if notebook 35 was run) | one score from the three | all of the above |

**Definitions (fixed before the run).** Evaluation set: **CAL-3** (200 injected events per symbol in 2024-08-02..08-31) plus the genuine August bars; the final models of `declared_final_set_v1.json`; T1c at 2 episodes/day is the main operating point (1 and 4 also shown).
- *Recall*: share of events with at least one alarm inside the event, macro over families, mean of BTC/ETH, event-bootstrap 95% interval.
- *Genuine episodes/day*: alarm episodes (gap ≤ 5 bars) per day on the genuine August bars with the same rule (T1).
- *ROC-AUC and PR-AUC* (bar level): positives = bars inside events of the family group; negatives = clean bars (no non-clean bar in the previous 16 bars) of the same series. PR-AUC depends on prevalence, so the prevalence is printed next to it.
- *Detection delay*: minutes from the event start to the first alarm inside the event (detected events only; median and 90th percentile).
- *Standard-amplitude recall*: recall on events with amplitude ≥ 1.0 (the generator base amplitude; the range is 0.5–2.0, log-uniform). This is reported next to, never instead of, the full recall.
- TEST-B / TEST-3 are untouched. Nothing is fitted or selected here.

In [ ]:
import os, json, glob, time, hashlib, warnings, sys
from pathlib import Path
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab -> using TRADESENSE_BASE.')
from sklearn.metrics import roc_auc_score, average_precision_score
BASE = Path(os.environ.get('TRADESENSE_BASE', '/content/drive/MyDrive/research datasets'))
L3 = BASE/'layer3_v3'; OUT = BASE/'models_b2_v1'
SYMBOLS = ['BTCUSDT', 'ETHUSDT']; NBOOT = 200 if os.environ.get('TS_MOCK') == '1' else 1000
LINES = []
def say(s=''): print(s); LINES.append(str(s))
def sha_file(p, n=16): return hashlib.sha256(Path(p).read_bytes()).hexdigest()[:n]
hf = L3/'protocol_bar_models_v2.sha256'; pf = L3/'protocol_bar_models_v2.md'
assert hf.exists() and pf.exists(), 'run notebook 27 first'
ph = hashlib.sha256(pf.read_text().encode()).hexdigest(); assert ph == hf.read_text().split()[0], 'frozen protocol B2 was changed'
assert (OUT/'declared_final_set_v1.json').exists(), 'run notebook 31 to the end first'
say('protocol B2 intact, sha256 ' + ph[:16])

Mounted at /content/drive
protocol B2 intact, sha256 443f99d5faf5929f


## 1 · Library (same file as notebooks 28-37)

In [ ]:
_old_lib = (OUT/'b2_lib.py').read_text() if (OUT/'b2_lib.py').exists() else None
open("b2_lib.py", "w").write(r'''
"""TradeSense C03 - Protocol B2 shared library: normalisation N0/N1, conditional autoencoder (AE-ctx), burden thresholds T0/T1, recall."""
import numpy as np, pandas as pd, torch, torch.nn as nn, time, json, hashlib
from numpy.lib.stride_tricks import sliding_window_view

FEATS = ['log_returns', 'trade_flow_imbalance', 'hl_spread_ratio', 'rush_order_density', 'volume_acceleration', 'large_trade_count']
LOGC = [3, 5]; WIN7 = 10080; MINP = 1440; CTX = 15; WARM = 30; GAPM = 5; CLIP = 20.0
BUDGETS = [1, 2, 4]; PRIMARY = 2; QGRID = [0.99, 0.993, 0.995, 0.997, 0.998, 0.999, 0.9993, 0.9995, 0.9997, 0.9998, 0.9999, 0.99995]
FAMS = {1: 'pump_and_dump_like', 2: 'fake_breakout_like', 3: 'abnormal_volume_burst', 4: 'trade_flow_imbalance', 5: 'wash_like_aggregate_proxy',
        6: 'liquidity_stress_proxy', 7: 'momentum_ignition_like', 8: 'slow_accumulation_like'}


# ---------------------------------------------------------------- normalisation
def to_x(F):
    x = np.asarray(F[FEATS] if hasattr(F, 'columns') else F, dtype='float64').copy()
    for j in LOGC: x[:, j] = np.log1p(x[:, j])
    return x


def fit_n0(F_train):
    """median / (IQR/1.349) on log1p-transformed Layer 1 TRAIN bars (caller passes bars >= 15 only)."""
    x = to_x(F_train); med = np.median(x, axis=0); q75, q25 = np.percentile(x, [75, 25], axis=0); return {'median': med, 'scale': (q75 - q25) / 1.349}


def normalise(F, mode, n0):
    x = to_x(F)
    if mode == 'N0': med, scl = n0['median'][None, :], n0['scale'][None, :]
    else:
        df = pd.DataFrame(x); r = df.rolling(WIN7, min_periods=MINP)
        med = r.median().shift(1).to_numpy(); q75 = r.quantile(0.75).shift(1).to_numpy(); q25 = r.quantile(0.25).shift(1).to_numpy()
        scl = np.maximum((q75 - q25) / 1.349, 0.1 * n0['scale'][None, :])
        bad = np.isnan(med) | np.isnan(scl); med = np.where(bad, n0['median'][None, :], med); scl = np.where(bad, n0['scale'][None, :], scl)
    return np.clip((x - med) / scl, -CLIP, CLIP)


def context(z):
    """causal context: mean and std (ddof 1) of the previous CTX bars; rows < CTX are zero-filled and never used."""
    n = len(z); m = np.zeros_like(z); s = np.zeros_like(z)
    w = sliding_window_view(z, CTX, axis=0)[: n - CTX]            # window k = bars k..k+14 -> bar k+15
    m[CTX:] = w.mean(axis=2); s[CTX:] = w.std(axis=2, ddof=1); return m, s


# ---------------------------------------------------------------- conditional autoencoder
class AECtx(nn.Module):
    def __init__(self, d=6, b=3):
        super().__init__(); self.enc = nn.Sequential(nn.Linear(d + 2 * d, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, b))
        self.dec = nn.Sequential(nn.Linear(b + 2 * d, 32), nn.ReLU(), nn.Linear(32, 64), nn.ReLU(), nn.Linear(64, d))

    def forward(self, z, cm, cs):
        c = torch.cat([cm, cs], 1); return self.dec(torch.cat([self.enc(torch.cat([z, c], 1)), c], 1))


def train_ae(z, idx, seed, max_epochs=150, patience=10, batch=512, log=None):
    """z normalised genuine series; idx = bar indices allowed for fitting (valid, inside Layer 1 TRAIN). 90% fit / 10% early stop, 15-bar gap."""
    torch.manual_seed(seed); rng = np.random.default_rng(seed); cm, cs = context(z)
    idx = np.sort(idx); cut = int(0.9 * len(idx)); fit = idx[:cut]; val = idx[idx > idx[cut] + CTX]
    T = lambda a: torch.tensor(a, dtype=torch.float32)
    Zf, Mf, Sf = T(z[fit]), T(cm[fit]), T(cs[fit]); Zv, Mv, Sv = T(z[val]), T(cm[val]), T(cs[val])
    net = AECtx(); opt = torch.optim.Adam(net.parameters(), lr=1e-3, weight_decay=1e-5); best, bad, state = 1e18, 0, None
    for ep in range(max_epochs):
        net.train(); perm = torch.tensor(rng.permutation(len(fit)))
        for i in range(0, len(perm), batch):
            b = perm[i:i + batch]; opt.zero_grad(); loss = ((net(Zf[b], Mf[b], Sf[b]) - Zf[b]) ** 2).mean(); loss.backward(); opt.step()
        net.eval()
        with torch.no_grad(): v = float(((net(Zv, Mv, Sv) - Zv) ** 2).mean())
        if v < best - 1e-6: best, bad, state = v, 0, {k: x.clone() for k, x in net.state_dict().items()}
        else:
            bad += 1
            if bad >= patience: break
    net.load_state_dict(state); net.eval(); return net, {'epochs': ep + 1, 'best_val': best, 'fit_rows': int(len(fit)), 'val_rows': int(len(val)), 'fit_min': int(fit.min()), 'fit_max': int(fit.max())}


def ae_errors(net, z):
    cm, cs = context(z); T = lambda a: torch.tensor(a, dtype=torch.float32)
    with torch.no_grad(): rec = net(T(z), T(cm), T(cs)).numpy()
    return (z - rec) ** 2


def p99_scale(err, idx): return np.percentile(err[idx], 99, axis=0) + 1e-12


def ae_columns(err, scale):
    """8 columns: fmax score, fmean score, 6 per-feature errors / TRAIN p99 scale."""
    r = err / scale[None, :]; return np.column_stack([r.max(axis=1), err.mean(axis=1), r])


# ---------------------------------------------------------------- burden: episodes, thresholds, recall
def episodes(alarm, gap=GAPM):
    a = np.flatnonzero(alarm)
    if len(a) == 0: return 0
    return int(1 + np.sum(np.diff(a) > gap))


def eps_per_day(alarm, valid):
    a = alarm & valid; days = valid.sum() / 1440.0
    return episodes(a) / max(days, 1e-9)


def valid_mask(n, lo, hi): m = np.zeros(n, bool); m[max(lo, WARM):hi] = True; return m


def t0_threshold(score, valid, budget):
    """lowest quantile-grid threshold (descending probability grid, lowering the threshold step by step) with episodes/day <= budget."""
    s = score[valid]; best = None
    for q in sorted(QGRID, reverse=True):
        thr = np.quantile(s, q)
        if eps_per_day(score > thr, valid) <= budget: best = thr
        else: break
    return best if best is not None else np.quantile(s, max(QGRID))


def rolling_thr(score, q):
    return pd.Series(score).rolling(WIN7, min_periods=MINP).quantile(q).shift(1).to_numpy()


def t1_q(score, valid, budget):
    """smallest quantile level q (lowest rolling threshold) with episodes/day <= budget on the genuine DEV bars."""
    for q in QGRID:
        thr = rolling_thr(score, q); alarm = (score > np.where(np.isnan(thr), np.inf, thr))
        if eps_per_day(alarm, valid) <= budget: return q
    return QGRID[-1]


def alarms(score, rule, par):
    if rule == 'T0': return score > par
    thr = rolling_thr(score, par); return score > np.where(np.isnan(thr), np.inf, thr)


def recall_by_family(alarm, ev, seg_ok=None):
    """ev rows: t0, dur, family, (replica); detected if an alarm falls in [t0, t0+dur-1]."""
    c = np.concatenate([[0], np.cumsum(alarm.astype(np.int64))]); out = {}
    for fam, sub in ev.groupby('family'):
        hit = [(c[r.t0 + r.dur] - c[r.t0]) > 0 for r in sub.itertuples()]; out[int(fam)] = (int(np.sum(hit)), int(len(hit)))
    return out


def bootstrap_ci(hits, n=1000, seed=0):
    rng = np.random.default_rng(seed); h = np.asarray(hits, float)
    if len(h) == 0: return (np.nan, np.nan)
    b = [h[rng.integers(0, len(h), len(h))].mean() for _ in range(n)]; return (float(np.percentile(b, 2.5)), float(np.percentile(b, 97.5)))


def macro(rec, fams):
    v = [rec[f][0] / rec[f][1] for f in fams if f in rec and rec[f][1] > 0]; return float(np.mean(v)) if v else float('nan')


def sha_bytes(b): return hashlib.sha256(b).hexdigest()

''')

import sys, importlib; sys.path.insert(0, '.'); import b2_lib as B; importlib.reload(B); (OUT/'b2_lib.py').write_text(Path('b2_lib.py').read_text())
say('library sha256 ' + sha_file('b2_lib.py'))
assert _old_lib is None or _old_lib.strip() == Path('b2_lib.py').read_text().strip(), 'b2_lib.py differs from the copy written by notebook 28'

library sha256 1b0b46630763c944


## 2 · Data (saved scores, CAL-3 events and labels)

In [ ]:
DF = json.load(open(OUT/'declared_final_set_v1.json')); MODELS = list(DF['models']); FNORM = {m: DF['models'][m]['normalisation'] for m in MODELS}
FILES = {'AE-ctx fmax': 'ae_ctx', 'XGBoost v2': 'xgb_v2', 'Transformer v2': 'transformer_v2'}
AOLEN = 31 * 1440; CAL_LO = 1440; PRIM = B.PRIMARY; BUD = B.BUDGETS; WARM = B.WARM; OLD, NEW = [1, 2, 3, 4, 5, 6], [7, 8]; ALLF = OLD + NEW
L1V = B.valid_mask(129600, 90720, 129600); CALV = B.valid_mask(AOLEN, CAL_LO, AOLEN)
def nanw(a): a = np.asarray(a, 'float64').copy(); a[:B.WARM] = np.nan; return a
SC = {}; EV = {}; LAB = {}; NAMES = list(MODELS)
for s in SYMBOLS:
    EV[s] = pd.read_csv(L3/f'l3v3_{s}_CAL_events.csv'); f = pd.read_parquet(L3/f'l3v3_{s}_CAL.parquet', columns=['label', 'in_seg', 'replica']); g = f[f.replica == 0].iloc[:AOLEN]; LAB[s] = (g.label.to_numpy().astype(int), g.in_seg.to_numpy().astype(bool))
    for m in MODELS:
        z = np.load(OUT/'scores'/f'{FILES[m]}_{s}_{FNORM[m]}.npz'); assert not any('TEST' in k for k in z.files); d = {}
        for k in ['GEN_L1', 'GEN_AO', 'CAL_r0']:
            x = z[k]; x = x[:, 0] if x.ndim == 2 else x; d[k] = nanw(x[:AOLEN] if k.startswith('CAL') else x)
        SC[(m, s)] = d
FUSION = (OUT/'declared_fusion_v0.json').exists()
if FUSION:
    DFU = json.load(open(OUT/'declared_fusion_v0.json')); NAMES.append('Fusion v0')
    for s in SYMBOLS:
        z = np.load(OUT/'scores'/f'fusion_v0_{s}.npz'); SC[('Fusion v0', s)] = {'GEN_AO': z['GEN_AO'].astype('float64'), 'CAL_r0': z['CAL_r0'].astype('float64')}
say('models in the scorecard: ' + ', '.join(NAMES) + ('' if FUSION else '  (fusion not found: run notebook 35 to add it)'))
for s in SYMBOLS: assert EV[s].t0.min() >= CAL_LO and (EV[s].t0 + EV[s].dur).max() <= AOLEN
def thr_roll(g, q): t = B.rolling_thr(g, q); return np.where(np.isnan(t), np.inf, t)
def alarms(m, s, bud):
    """T1c: returns (alarm on CAL-3 series, alarm on genuine August series)."""
    if m == 'Fusion v0': q = DFU['q_T1_by_budget'][str(bud)][s]
    else: q = B.t1_q(SC[(m, s)]['GEN_L1'], L1V, bud)
    thr = thr_roll(SC[(m, s)]['GEN_AO'], q); return SC[(m, s)]['CAL_r0'] > thr, SC[(m, s)]['GEN_AO'] > thr
META = pd.concat([pd.DataFrame({'sym': s, 'fam': EV[s].family.to_numpy(), 'amp': EV[s].amp.to_numpy(), 't0': EV[s].t0.to_numpy(), 'dur': EV[s].dur.to_numpy()}) for s in SYMBOLS], ignore_index=True)
def event_hits(a, ev):
    c = np.concatenate([[0], np.cumsum(np.nan_to_num(a).astype(np.int64))]); t0 = ev.t0.to_numpy(); du = ev.dur.to_numpy(); return (c[t0 + du] - c[t0]) > 0
def delays(a, ev):
    out = []
    for r in ev.itertuples():
        seg = np.nan_to_num(a[r.t0:r.t0 + r.dur]).astype(bool); out.append(int(np.argmax(seg)) if seg.any() else -1)
    return np.array(out)
def groups(fams, sel=None):
    G = []
    for s in SYMBOLS:
        gs = []
        for f in fams:
            k = ((META.sym == s) & (META.fam == f)).to_numpy().copy()
            if sel is not None: k &= sel
            if k.any(): gs.append(np.flatnonzero(k))
        G.append(gs)
    return G
def macro(h, fams, sel=None): return float(np.mean([np.mean([h[i].mean() for i in gs]) for gs in groups(fams, sel)]))
def boot(h, fams, sel=None, n=None, seed=0):
    n = n or NBOOT; G = groups(fams, sel); rg = np.random.default_rng(seed); out = np.zeros(n)
    for i in range(n): out[i] = np.mean([np.mean([h[idx[p]].mean() for idx in gs for p in [rg.integers(0, len(idx), len(idx))]]) for gs in G])
    return f'{np.percentile(out, 2.5):.2f}-{np.percentile(out, 97.5):.2f}'
def fam_rates(h): return pd.DataFrame({'sym': META.sym, 'fam': META.fam, 'h': h}).groupby(['sym', 'fam']).h.mean().groupby('fam').mean()
def rank_metrics(m, fams):
    ys, ss, pos_n = [], [], 0; tot = 0
    for s in SYMBOLS:
        lab, inseg = LAB[s]; n = len(lab); idx = np.arange(n); c = np.concatenate([[0], np.cumsum((lab != 0).astype(np.int64))])
        clean = ((c[idx + 1] - c[np.maximum(idx - 16, 0)]) == 0) & inseg & (idx >= WARM); pos = np.isin(lab, fams) & inseg & (idx >= WARM); sc = SC[(m, s)]['CAL_r0']; ok = ~np.isnan(sc); mk = (pos | clean) & ok
        ys.append(pos[mk]); ss.append(sc[mk])
    y = np.concatenate(ys); sc_ = np.concatenate(ss); return float(roc_auc_score(y, sc_)), float(average_precision_score(y, sc_)), float(y.mean())

models in the scorecard: AE-ctx fmax, XGBoost v2, Transformer v2, Fusion v0


## 3 · Unit tests

In [ ]:
# T01 hit rule against a plain loop; T02 delay definition; T03 ROC-AUC of a perfect and an inverted score; T04 bootstrap interval brackets the point estimate
rg = np.random.default_rng(0); ev0 = EV[SYMBOLS[0]]
for _ in range(5):
    al = rg.random(AOLEN) < 0.002; fast = event_hits(al, ev0); slow = np.array([al[r.t0:r.t0 + r.dur].any() for r in ev0.itertuples()]); assert (fast == slow).all()
say('PASS T01 event hit rule equals a plain loop')
a = np.zeros(AOLEN, bool); e1 = pd.DataFrame({'t0': [100, 500], 'dur': [20, 30]}); a[107] = True; a[510] = True; a[520] = True; assert list(delays(a, e1)) == [7, 10]; assert list(delays(np.zeros(AOLEN, bool), e1)) == [-1, -1]; say('PASS T02 detection delay = bars from the event start to the first alarm (-1 if missed)')
yy = np.array([0, 0, 1, 1]); assert roc_auc_score(yy, [0.1, 0.2, 0.8, 0.9]) == 1.0 and roc_auc_score(yy, [0.9, 0.8, 0.2, 0.1]) == 0.0; say('PASS T03 ROC-AUC conventions (perfect = 1, inverted = 0)')
m0 = NAMES[0]; al, _ = alarms(m0, SYMBOLS[0], PRIM); h = np.concatenate([event_hits(alarms(m0, s, PRIM)[0], EV[s]) for s in SYMBOLS]); pt = macro(h, OLD); lo, hi = map(float, boot(h, OLD).split('-')); assert lo - 0.03 <= pt <= hi + 0.03; say('PASS T04 bootstrap interval brackets the point estimate')

PASS T01 event hit rule equals a plain loop
PASS T02 detection delay = bars from the event start to the first alarm (-1 if missed)
PASS T03 ROC-AUC conventions (perfect = 1, inverted = 0)
PASS T04 bootstrap interval brackets the point estimate


## 4 · Scorecard

In [ ]:
ROWS = []; HITS = {}; DEL = {}; GEPS = {}
for m in NAMES:
    for bud in BUD:
        al = {s: alarms(m, s, bud) for s in SYMBOLS}; h = np.concatenate([event_hits(al[s][0], EV[s]) for s in SYMBOLS]); HITS[(m, bud)] = h
        GEPS[(m, bud)] = float(np.mean([B.eps_per_day(al[s][1], CALV) for s in SYMBOLS]))
        if bud == PRIM: DEL[m] = np.concatenate([delays(al[s][0], EV[s]) for s in SYMBOLS])
    h2 = HITS[(m, PRIM)]; std = (META.amp.to_numpy() >= 1.0); d = DEL[m]; det = d[d >= 0]; r_auc, r_ap, r_prev = rank_metrics(m, OLD); n_auc, n_ap, n_prev = rank_metrics(m, NEW); a_auc, a_ap, a_prev = rank_metrics(m, ALLF)
    ROWS.append({'model': m, 'genuine eps/day (2/day rule)': GEPS[(m, PRIM)], 'recall 6 old': macro(h2, OLD), 'old 95% CI': boot(h2, OLD), 'recall 2 unseen': macro(h2, NEW), 'unseen 95% CI': boot(h2, NEW), 'recall 8 families': macro(h2, ALLF),
                 'recall 8, amp>=1': macro(h2, ALLF, std), 'ROC-AUC 6 old': r_auc, 'PR-AUC 6 old': r_ap, 'prevalence 6 old': r_prev, 'ROC-AUC all 8': a_auc, 'median delay (min)': float(np.median(det)) if len(det) else np.nan, 'p90 delay (min)': float(np.percentile(det, 90)) if len(det) else np.nan})
SCORECARD = pd.DataFrame(ROWS); say('--- scorecard on CAL-3 (T1c, 2 episodes/day each; ROC/PR at bar level) ---'); say(SCORECARD.round(3).T.to_string(header=False))
BT = pd.DataFrame([{'model': m, 'budget/day': bud, 'genuine eps/day': GEPS[(m, bud)], 'recall 6 old': macro(HITS[(m, bud)], OLD), 'recall 2 unseen': macro(HITS[(m, bud)], NEW), 'recall 8, amp>=1': macro(HITS[(m, bud)], ALLF, (META.amp.to_numpy() >= 1.0))} for m in NAMES for bud in BUD])
say(''); say('--- recall at budgets 1 / 2 / 4 episodes per day ---'); say(BT.round(3).to_string(index=False))
FT = pd.DataFrame({m: fam_rates(HITS[(m, PRIM)]) for m in NAMES}); FT.index = [B.FAMS[i] for i in FT.index]; say(''); say('--- recall per family (2/day) ---'); say(FT.round(2).to_string())

--- scorecard on CAL-3 (T1c, 2 episodes/day each; ROC/PR at bar level) ---
model                         AE-ctx fmax  XGBoost v2  Transformer v2  Fusion v0
genuine eps/day (2/day rule)        1.433       1.167            1.75      1.567
recall 6 old                         0.22        0.71            0.73       0.78
old 95% CI                      0.18-0.26   0.67-0.75       0.69-0.77  0.74-0.81
recall 2 unseen                      0.16        0.75            0.61       0.74
unseen 95% CI                   0.10-0.24   0.67-0.83       0.51-0.70  0.65-0.83
recall 8 families                   0.205        0.72             0.7       0.77
recall 8, amp>=1                    0.271       0.815           0.822      0.869
ROC-AUC 6 old                        0.67       0.901           0.901      0.922
PR-AUC 6 old                        0.208       0.746           0.748      0.804
prevalence 6 old                    0.088       0.088           0.088      0.088
ROC-AUC all 8                     

## 5 · Reading per task and save

In [ ]:
say(''); say('--- reading per task (numbers from the tables above) ---')
def g(m, col): return SCORECARD[SCORECARD.model == m][col].iloc[0]
for m, task in [('AE-ctx fmax', 'novelty detection without labels'), ('XGBoost v2', 'known-pattern classification'), ('Transformer v2', 'multi-bar sequence patterns')]:
    if m in NAMES:
        chance = 0.02; say(f'{m} ({task}): recall 6 families {g(m, "recall 6 old"):.2f} {g(m, "old 95% CI")}, unseen families {g(m, "recall 2 unseen"):.2f}, at amplitude >= 1: {g(m, "recall 8, amp>=1"):.2f}; ROC-AUC {g(m, "ROC-AUC 6 old"):.2f}, PR-AUC {g(m, "PR-AUC 6 old"):.2f} (prevalence {g(m, "prevalence 6 old"):.3f}); delay median {g(m, "median delay (min)"):.0f} min; genuine {g(m, "genuine eps/day (2/day rule)"):.2f} episodes/day; chance recall about {chance}')
if 'Fusion v0' in NAMES: say(f'Fusion v0: recall 6 families {g("Fusion v0", "recall 6 old"):.2f} {g("Fusion v0", "old 95% CI")}, unseen {g("Fusion v0", "recall 2 unseen"):.2f}, at amplitude >= 1: {g("Fusion v0", "recall 8, amp>=1"):.2f}; genuine {g("Fusion v0", "genuine eps/day (2/day rule)"):.2f} episodes/day')
SCORECARD.to_csv(OUT/'scorecard_cal3.csv', index=False); BT.to_csv(OUT/'scorecard_cal3_budgets.csv', index=False); FT.to_csv(OUT/'scorecard_cal3_by_family.csv')
block = '\n'.join(LINES); (OUT/'b2_step9_share_block.txt').write_text(block)
print('\n===== SHARE BLOCK (attach models_b2_v1/b2_step9_share_block.txt, or copy below) =====\n' + block[-12000:])


--- reading per task (numbers from the tables above) ---
AE-ctx fmax (novelty detection without labels): recall 6 families 0.22 0.18-0.26, unseen families 0.16, at amplitude >= 1: 0.27; ROC-AUC 0.67, PR-AUC 0.21 (prevalence 0.088); delay median 5 min; genuine 1.43 episodes/day; chance recall about 0.02
XGBoost v2 (known-pattern classification): recall 6 families 0.71 0.67-0.75, unseen families 0.75, at amplitude >= 1: 0.82; ROC-AUC 0.90, PR-AUC 0.75 (prevalence 0.088); delay median 8 min; genuine 1.17 episodes/day; chance recall about 0.02
Transformer v2 (multi-bar sequence patterns): recall 6 families 0.73 0.69-0.77, unseen families 0.61, at amplitude >= 1: 0.82; ROC-AUC 0.90, PR-AUC 0.75 (prevalence 0.088); delay median 8 min; genuine 1.75 episodes/day; chance recall about 0.02
Fusion v0: recall 6 families 0.78 0.74-0.81, unseen 0.74, at amplitude >= 1: 0.87; genuine 1.57 episodes/day

===== SHARE BLOCK (attach models_b2_v1/b2_step9_share_block.txt, or copy below) =====
protocol B2 